# Modèle 2 — Analyse exploratoire de l'apport des signaux email

**Objectif :** Ce notebook répond directement à la problématique du mémoire :  
*« Les signaux d'engagement email apportent-ils un pouvoir prédictif additionnel au-delà des variables structurelles ? »*

**Méthodologie :**  
On isole les dossiers ayant un historique email réel (`est_dans_crm=1`), puis on compare deux modèles XGBoost entraînés dans des conditions identiques :
- **Modèle A** : features structurelles uniquement (canal, Flex, anticipation, fidélité…)
- **Modèle B** : features structurelles + features email (ouvertures, clics, désabonnements…)

Si PR-AUC(B) > PR-AUC(A) de manière significative, l'email apporte un signal additionnel.  
Sinon, les variables structurelles suffisent — résultat tout aussi intéressant scientifiquement.

**Positionnement :** Ce notebook constitue le *Tier 2* de la stratégie de modélisation en deux temps définie dans le mémoire. Le *Tier 1* (modèle global sur ~310k dossiers) a déjà montré via SHAP que les features email pèsent peu — mais sur un dataset dont 90.7% des dossiers n'ont aucun historique email. Le vrai test se fait ici, sur la population qui a effectivement reçu des emails.

---
## 1. Chargement et filtrage du sous-ensemble CRM

Je charge le dataset nettoyé produit par `EDA.ipynb` et je filtre uniquement les dossiers ayant un historique email dans Batch (`est_dans_crm=1`). C'est sur cette population que la comparaison A vs B a du sens — inutile de tester l'apport email sur des dossiers qui n'en ont pas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Chargement du dataset propre ─────────────────────────────
CSV_PATH = 'data/dataset_annulation_clean.csv'
df_full = pd.read_csv(CSV_PATH, encoding='utf-8-sig', low_memory=False)

# ── Filtrage : uniquement les dossiers avec historique email ──
df_crm = df_full[df_full['est_dans_crm'] == 1].copy()

print(f"Dataset complet   : {len(df_full):,} dossiers")
print(f"Sous-ensemble CRM : {len(df_crm):,} dossiers "
      f"({len(df_crm)/len(df_full)*100:.1f}%)")
print(f"\nTaux d'annulation :")
print(f"  Global : {df_full['y_annulation'].mean()*100:.2f}%")
print(f"  CRM    : {df_crm['y_annulation'].mean()*100:.2f}%")
print(f"\nAnnulations dans le sous-ensemble CRM : "
      f"{df_crm['y_annulation'].sum():,}")

---
## 2. Vérification du signal email

Avant de construire les modèles, je vérifie que les features email ont effectivement de la variance dans ce sous-ensemble. Sur le dataset global, 90% des dossiers ont tout à 0 pour l'email — ici, la situation devrait être différente puisque ces clients ont reçu des campagnes.

Je regarde trois choses :
1. Les **statistiques descriptives** (moyenne, écart-type, % de zéros)
2. Le **taux d'annulation** selon que le client a interagi ou non avec les emails
3. Les **corrélations** entre chaque feature email et la cible `y_annulation`

In [ ]:
# ── Définition des features email ─────────────────────────────
EMAIL_FEATURES = [
    'nb_clics_90j', 'nb_ouvertures_90j', 'nb_desabo_90j',
    'nb_campagnes_recues', 'nb_campagnes_cliquees',
    'nb_urls_distinctes_cliquees', 'taux_clic_sur_ouverture',
    'a_interagi_email', 'recence_email_jours',
]

email_presentes = [c for c in EMAIL_FEATURES if c in df_crm.columns]
email_absentes  = [c for c in EMAIL_FEATURES if c not in df_crm.columns]

if email_absentes:
    print(f"⚠️ Features email absentes du dataset : {email_absentes}")

# ── Statistiques descriptives ────────────────────────────────
print("=== STATISTIQUES DES FEATURES EMAIL (sous-ensemble CRM) ===\n")
stats = df_crm[email_presentes].describe().T[['mean', 'std', 'min', 'max']]
stats['pct_zero'] = (df_crm[email_presentes] == 0).mean().values * 100
stats['pct_null'] = df_crm[email_presentes].isnull().mean().values * 100
print(stats.round(2).to_string())

# ── Taux d'annulation selon l'interaction email ──────────────
if 'a_interagi_email' in df_crm.columns:
    print("\n\nTaux d'annulation selon l'interaction email :")
    res = (df_crm.groupby('a_interagi_email')['y_annulation']
                 .agg(nb='count', taux='mean')
                 .assign(taux=lambda x: (x['taux']*100).round(2)))
    res.index = ['Pas d\'interaction (0)', 'A interagi (1)']
    print(res.to_string())

# ── Corrélations email → annulation ──────────────────────────
print("\n\nCorrélations features email → y_annulation :")
corrs = df_crm[email_presentes + ['y_annulation']].corr()['y_annulation']
corrs = corrs.drop('y_annulation').sort_values(key=abs, ascending=False)
for feat, val in corrs.items():
    signal = '📊' if abs(val) > 0.05 else '  '
    print(f"  {signal} {feat:<35} {val:+.4f}")

---
## 3. Définition des deux groupes de features

Le protocole est simple : deux modèles identiques (même algorithme, mêmes hyperparamètres, même split), la seule différence étant les features en entrée.

- **Modèle A** = variables structurelles seules (canal, Flex, anticipation, fidélité, distance, composition du groupe…)
- **Modèle B** = variables structurelles + les 9 features email

Si le Modèle B fait mieux, c'est grâce à l'email. Si les scores sont identiques, l'email n'apporte rien de plus que ce que les variables structurelles captent déjà.

In [ ]:
# ── Features structurelles (sans aucune feature email) ────────
FEATURES_STRUCT_NUM = [
    'anticipation_jours', 'duree_sejour',
    'dossier_nb_pax_total', 'dossier_nb_pax_adultes',
    'nb_mineur', 'nb_bebe', 'mois_resa',
    'nb_dossiers_anterieurs', 'est_client_vip',
    'est_assure_annulation',
    'nb_produits_total',
    'assure_x_anticip', 'est_solo',
    'distance_km',
]

FEATURES_STRUCT_CAT = [
    'canal', 'groupe_fournisseur', 'periode_depart', 'periode_vacances',
    'type_produit', 'device_resa', 'theme_station', 'region_destination',
    'type_hebergement', 'cond_annulation',
    'anticipation_tranche',
]

# ── Features email ───────────────────────────────────────────
FEATURES_EMAIL = [c for c in email_presentes if c in df_crm.columns]

# ── Filtrer aux colonnes réellement présentes ────────────────
FEATURES_STRUCT_NUM = [c for c in FEATURES_STRUCT_NUM if c in df_crm.columns]
FEATURES_STRUCT_CAT = [c for c in FEATURES_STRUCT_CAT if c in df_crm.columns]

# Modèle A : structurelles uniquement
FEATURES_A = FEATURES_STRUCT_NUM + FEATURES_STRUCT_CAT
# Modèle B : structurelles + email
FEATURES_B = FEATURES_STRUCT_NUM + FEATURES_EMAIL + FEATURES_STRUCT_CAT

print(f"Modèle A (structurel seul) : {len(FEATURES_A)} features")
print(f"  NUM : {len(FEATURES_STRUCT_NUM)}")
print(f"  CAT : {len(FEATURES_STRUCT_CAT)}")
print(f"\nModèle B (structurel + email) : {len(FEATURES_B)} features")
print(f"  NUM : {len(FEATURES_STRUCT_NUM)} + {len(FEATURES_EMAIL)} email "
      f"= {len(FEATURES_STRUCT_NUM) + len(FEATURES_EMAIL)}")
print(f"  CAT : {len(FEATURES_STRUCT_CAT)}")
print(f"\nFeatures email ajoutées dans B :")
for f in FEATURES_EMAIL:
    print(f"  + {f}")

---
## 4. Entraînement et comparaison des deux modèles

Protocole identique au modèle global :
- Split stratifié 80/20
- XGBoost avec `scale_pos_weight` (pondération native, pas de SMOTE)
- Mêmes hyperparamètres de base pour les deux modèles

La métrique principale de comparaison est le **PR-AUC** (Average Precision), la plus adaptée à un dataset déséquilibré.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix,
                             roc_curve, precision_recall_curve)
import time

# ── Split stratifié ──────────────────────────────────────────
y_crm = df_crm['y_annulation']

X_train_crm, X_test_crm, y_train_crm, y_test_crm = train_test_split(
    df_crm, y_crm, test_size=0.20, random_state=42, stratify=y_crm
)

n_pos = y_train_crm.sum()
n_neg = len(y_train_crm) - n_pos
ratio_poids_crm = n_neg / n_pos

print(f"Train CRM : {len(X_train_crm):,} ({y_train_crm.mean()*100:.2f}% annul.)")
print(f"Test  CRM : {len(X_test_crm):,} ({y_test_crm.mean()*100:.2f}% annul.)")
print(f"Pondération (scale_pos_weight) : {ratio_poids_crm:.2f}")


def creer_pipeline(features_num, features_cat, ratio_poids):
    """Crée un pipeline XGBoost avec preprocessing intégré."""
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='inconnu')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                                   unknown_value=-1)),
    ])
    preprocessor = ColumnTransformer([
        ('num', num_pipe, features_num),
        ('cat', cat_pipe, features_cat),
    ])
    return Pipeline([
        ('prep',  preprocessor),
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=ratio_poids,
            random_state=42,
            eval_metric='aucpr',
            n_jobs=-1,
        )),
    ])


# ── Entraînement des deux modèles ────────────────────────────
resultats = {}

configs = {
    'Modèle A (structurel)': {
        'features_num': FEATURES_STRUCT_NUM,
        'features_cat': FEATURES_STRUCT_CAT,
        'all_features':  FEATURES_A,
    },
    'Modèle B (structurel + email)': {
        'features_num': FEATURES_STRUCT_NUM + FEATURES_EMAIL,
        'features_cat': FEATURES_STRUCT_CAT,
        'all_features':  FEATURES_B,
    },
}

for nom, cfg in configs.items():
    print(f"\n{'='*55}")
    print(f"  {nom}")
    print(f"{'='*55}")

    t0 = time.time()
    pipe = creer_pipeline(cfg['features_num'], cfg['features_cat'],
                          ratio_poids_crm)
    pipe.fit(X_train_crm[cfg['all_features']], y_train_crm)

    y_proba = pipe.predict_proba(X_test_crm[cfg['all_features']])[:, 1]
    y_pred  = pipe.predict(X_test_crm[cfg['all_features']])
    duree = time.time() - t0

    metrics = {
        'pipeline':   pipe,
        'proba':      y_proba,
        'pred':       y_pred,
        'features':   cfg['all_features'],
        'AUC-ROC':    roc_auc_score(y_test_crm, y_proba),
        'PR-AUC':     average_precision_score(y_test_crm, y_proba),
        'F1':         f1_score(y_test_crm, y_pred),
        'Précision':  precision_score(y_test_crm, y_pred),
        'Rappel':     recall_score(y_test_crm, y_pred),
        'Durée (s)':  round(duree, 1),
    }
    resultats[nom] = metrics

    print(f"  AUC-ROC   : {metrics['AUC-ROC']:.4f}")
    print(f"  PR-AUC    : {metrics['PR-AUC']:.4f}")
    print(f"  F1        : {metrics['F1']:.4f}")
    print(f"  Précision : {metrics['Précision']:.4f}")
    print(f"  Rappel    : {metrics['Rappel']:.4f}")
    print(f"  Durée     : {metrics['Durée (s)']:.1f}s")


# ── Tableau comparatif ───────────────────────────────────────
print(f"\n{'='*60}")
print(f"  COMPARAISON : STRUCTUREL vs STRUCTUREL + EMAIL")
print(f"{'='*60}")
print(f"Référence PR-AUC (aléatoire) : {y_test_crm.mean():.4f}\n")

df_comp = pd.DataFrame({
    nom: {k: v for k, v in m.items()
          if k in ['AUC-ROC','PR-AUC','F1','Précision','Rappel']}
    for nom, m in resultats.items()
}).T

delta = df_comp.iloc[1] - df_comp.iloc[0]
df_comp.loc['Δ (B - A)'] = delta
print(df_comp.round(4).to_string())

delta_pr = delta['PR-AUC']
print(f"\n{'─'*60}")
if delta_pr > 0.01:
    print(f"  ✔ L'email apporte un gain de PR-AUC de +{delta_pr:.4f}")
    print(f"    → Signal additionnel détecté, même s'il reste modeste.")
elif delta_pr > 0:
    print(f"  ≈ L'email apporte un gain marginal de +{delta_pr:.4f}")
    print(f"    → Gain trop faible pour être considéré significatif.")
else:
    print(f"  ✗ L'email n'apporte aucun gain (Δ = {delta_pr:.4f})")
    print(f"    → Les variables structurelles suffisent.")
print(f"{'─'*60}")

---
## 5. Courbes ROC et Précision-Rappel comparatives

Les deux courbes côte à côte permettent de visualiser si l'ajout des features email déplace la courbe du Modèle B au-dessus de celle du Modèle A. Sur la courbe PR (droite), même un petit écart vertical est significatif dans un contexte déséquilibré.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

couleurs = {
    'Modèle A (structurel)':           '#3498db',
    'Modèle B (structurel + email)':   '#e74c3c',
}

# ── Courbe ROC ───────────────────────────────────────────────
for nom, res in resultats.items():
    fpr, tpr, _ = roc_curve(y_test_crm, res['proba'])
    ax1.plot(fpr, tpr, color=couleurs[nom], lw=2,
             label=f"{nom} (AUC={res['AUC-ROC']:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Aléatoire (0.5)')
ax1.set_xlabel('Faux positifs')
ax1.set_ylabel('Vrais positifs')
ax1.set_title('Courbe ROC — sous-ensemble CRM')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Courbe Précision-Rappel ──────────────────────────────────
ref_pr = y_test_crm.mean()

for nom, res in resultats.items():
    prec_c, rec_c, _ = precision_recall_curve(y_test_crm, res['proba'])
    ax2.plot(rec_c, prec_c, color=couleurs[nom], lw=2,
             label=f"{nom} (AP={res['PR-AUC']:.3f})")

ax2.axhline(ref_pr, ls='--', color='k', lw=1.5,
            label=f'Référence ({ref_pr:.3f})')
ax2.set_xlabel('Rappel')
ax2.set_ylabel('Précision')
ax2.set_title('Courbe Précision-Rappel — sous-ensemble CRM')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Apport des features email : Modèle A (structurel) vs '
             'Modèle B (structurel + email)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('figures/17_comparaison_email_roc_pr.png',
            dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Interprétabilité SHAP — Position des features email

L'analyse SHAP sur le Modèle B permet de répondre à une question précise : **où se classent les features email dans le ranking d'importance ?**

Si elles apparaissent dans le top 10, leur signal est réel mais insuffisamment capté par le dataset actuel. Si elles sont en queue de classement, les variables structurelles dominent totalement.

In [ ]:
import shap

# ── SHAP sur le Modèle B (structurel + email) ────────────────
pipe_B = resultats['Modèle B (structurel + email)']['pipeline']
xgb_B  = pipe_B.named_steps['model']
X_test_B_transformed = pipe_B.named_steps['prep'].transform(
    X_test_crm[FEATURES_B]
)

feature_names_B = FEATURES_STRUCT_NUM + FEATURES_EMAIL + FEATURES_STRUCT_CAT

print("Calcul SHAP sur le Modèle B (structurel + email)...")
explainer_B   = shap.TreeExplainer(xgb_B)
shap_values_B = explainer_B.shap_values(X_test_B_transformed)
print(f"✔ SHAP calculé sur {X_test_B_transformed.shape[0]:,} observations")

# ── Summary plot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values_B, X_test_B_transformed,
                  feature_names=feature_names_B,
                  max_display=25, show=False)
plt.title('SHAP — Modèle B (structurel + email) — sous-ensemble CRM',
          fontsize=12)
plt.tight_layout()
plt.savefig('figures/18_shap_modele_email.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Classement complet avec marquage email ───────────────────
shap_importance_B = np.abs(shap_values_B).mean(axis=0)
ranking = np.argsort(shap_importance_B)[::-1]

print(f"\n{'='*60}")
print(f"  CLASSEMENT SHAP — POSITION DES FEATURES EMAIL")
print(f"{'='*60}")

for rank, idx in enumerate(ranking, 1):
    nom_feat = feature_names_B[idx]
    imp = shap_importance_B[idx]
    marqueur = ' 📧' if nom_feat in FEATURES_EMAIL else ''
    print(f"  {rank:>2}. {nom_feat:<35} "
          f"|SHAP| = {imp:.4f}{marqueur}")

# ── Focus email ──────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  FOCUS — Features email uniquement :")
print(f"{'─'*60}")
for feat in FEATURES_EMAIL:
    idx_f = feature_names_B.index(feat)
    imp   = shap_importance_B[idx_f]
    rang  = list(ranking).index(idx_f) + 1
    print(f"  #{rang:<3} {feat:<35} |SHAP| = {imp:.4f}")

---
## 7. Test de significativité (bootstrap)

Le delta PR-AUC observé entre les modèles A et B pourrait être dû au hasard de l'échantillonnage. Pour le vérifier, je fais un **test bootstrap** :
- Je rééchantillonne le test set 1 000 fois (tirage avec remise)
- À chaque itération, je calcule le delta PR-AUC(B) − PR-AUC(A)
- Si l'intervalle de confiance à 95% est entièrement au-dessus de 0, le gain est significatif

C'est une méthode robuste et non paramétrique, parfaitement adaptée à ce contexte.

In [ ]:
proba_A = resultats['Modèle A (structurel)']['proba']
proba_B = resultats['Modèle B (structurel + email)']['proba']

n_bootstrap = 1000
deltas = []
np.random.seed(42)

y_test_arr = y_test_crm.values

for _ in range(n_bootstrap):
    idx = np.random.choice(len(y_test_arr), size=len(y_test_arr), replace=True)
    y_b = y_test_arr[idx]

    if y_b.sum() == 0 or y_b.sum() == len(y_b):
        continue

    pr_a = average_precision_score(y_b, proba_A[idx])
    pr_b = average_precision_score(y_b, proba_B[idx])
    deltas.append(pr_b - pr_a)

deltas = np.array(deltas)

ci_low  = np.percentile(deltas, 2.5)
ci_high = np.percentile(deltas, 97.5)
pct_positive = (deltas > 0).mean() * 100

print(f"{'='*55}")
print(f"  TEST BOOTSTRAP — Significativité du delta PR-AUC")
print(f"{'='*55}")
print(f"  Nombre d'itérations    : {n_bootstrap}")
print(f"  Delta PR-AUC observé   : {delta_pr:+.4f}")
print(f"  IC 95%                 : [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"  % itérations Δ > 0     : {pct_positive:.1f}%")
print(f"{'─'*55}")

if ci_low > 0:
    print(f"  ✔ Le gain est SIGNIFICATIF (IC entièrement > 0)")
    print(f"    → L'email apporte un pouvoir prédictif additionnel.")
elif pct_positive > 80:
    print(f"  ≈ Le gain est TENDANCIEL ({pct_positive:.0f}% > 0, mais IC inclut 0)")
    print(f"    → Signal faible, probablement amplifié avec plus de données.")
else:
    print(f"  ✗ Le gain n'est PAS significatif ({pct_positive:.0f}% > 0)")
    print(f"    → Les variables structurelles suffisent ; l'email n'apporte")
    print(f"      pas de pouvoir prédictif additionnel dans l'état actuel.")

# ── Histogramme du delta ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(deltas, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
ax.axvline(0, color='k', ls='--', lw=2, label='Δ = 0 (pas de gain)')
ax.axvline(delta_pr, color='#e74c3c', ls='-', lw=2,
           label=f'Δ observé ({delta_pr:+.4f})')
ax.axvline(ci_low, color='#95a5a6', ls=':', lw=1.5,
           label=f'IC 95% [{ci_low:+.4f}, {ci_high:+.4f}]')
ax.axvline(ci_high, color='#95a5a6', ls=':', lw=1.5)
ax.set_xlabel('Delta PR-AUC (Modèle B − Modèle A)')
ax.set_ylabel('Fréquence')
ax.set_title("Bootstrap : distribution du gain PR-AUC apporté par l'email")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/19_bootstrap_delta_email.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Synthèse et conclusion

Cette section résume l'ensemble des résultats de l'analyse exploratoire et formule la réponse à la problématique du mémoire concernant l'apport des signaux email.

In [ ]:
pr_a = resultats['Modèle A (structurel)']['PR-AUC']
pr_b = resultats['Modèle B (structurel + email)']['PR-AUC']
delta_final = pr_b - pr_a

print("=" * 65)
print("  SYNTHÈSE — APPORT DES SIGNAUX EMAIL")
print("=" * 65)

print(f"""
┌─────────────────────────────────────────────────────────┐
│  Sous-ensemble analysé : est_dans_crm = 1               │
│  Dossiers              : {len(df_crm):,}                       │
│  Taux d'annulation     : {df_crm['y_annulation'].mean()*100:.2f}%                       │
├─────────────────────────────────────────────────────────┤
│  PR-AUC Modèle A (structurel seul)  : {pr_a:.4f}            │
│  PR-AUC Modèle B (struct. + email)  : {pr_b:.4f}            │
│  Delta                              : {delta_final:+.4f}            │
├─────────────────────────────────────────────────────────┤
│  Bootstrap IC 95%  : [{ci_low:+.4f}, {ci_high:+.4f}]               │
│  % itérations Δ>0  : {pct_positive:.1f}%                          │
└─────────────────────────────────────────────────────────┘
""")

print("INTERPRÉTATION POUR LE MÉMOIRE :\n")

if ci_low > 0:
    print("""Les signaux d'engagement email apportent un pouvoir prédictif
ADDITIONNEL statistiquement significatif. Cependant, le gain
reste modeste en valeur absolue, ce qui s'explique par :
  1. La couverture limitée de Batch (7 mois d'historique)
  2. La dominance des variables structurelles (canal, Flex,
     anticipation) qui captent déjà l'essentiel du signal

RECOMMANDATION : enrichir l'historique email (12-18 mois)
et réentraîner le modèle pour amplifier ce signal.""")
elif pct_positive > 80:
    print("""Les signaux email montrent une TENDANCE positive mais non
significative au seuil de 95%. Le signal existe mais est
trop faible pour être distingué du bruit avec les données
actuelles. Causes probables :
  1. Seulement 7 mois d'historique Batch
  2. Couverture de 9.3% des dossiers seulement
  3. Agrégation sur 90 jours qui lisse les signaux

RECOMMANDATION : prolonger la collecte email, affiner la
fenêtre d'agrégation (30j, 60j, 90j), et intégrer les
données GA4 quand le problème de région sera résolu.""")
else:
    print("""Les signaux email N'APPORTENT PAS de pouvoir prédictif
additionnel dans l'état actuel des données. Les variables
structurelles (canal, Flex, anticipation, fidélité) suffisent
à expliquer le risque d'annulation.

Ce résultat négatif est scientifiquement intéressant :
  1. Il montre que la décision d'annuler est principalement
     structurelle (conditions tarifaires, canal d'acquisition)
  2. Le comportement email post-réservation ne reflète pas
     (encore) l'intention d'annuler
  3. Avec seulement 7 mois de Batch et 9.3% de couverture,
     les données sont insuffisantes pour conclure définitivement

RECOMMANDATION : ne pas abandonner l'hypothèse email, mais
attendre 12-18 mois de données supplémentaires et intégrer
des signaux comportementaux plus riches (navigation web,
consultation page annulation) avant de conclure.""")

print("\n" + "─" * 65)
print("  Ce notebook constitue le Tier 2 de la stratégie de")
print("  modélisation — Analyse exploratoire : apport email")
print("─" * 65)